<a href="https://colab.research.google.com/github/KatharinaMaubach/Votes_to_Victory/blob/main/dashboard_electionresults.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
import pandas as pd

from google.colab import drive
drive.mount('/content/drive')

file_path = "/content/drive/MyDrive/TechLabs_Gruppe3/data_all.xlsx"
data_all = pd.read_excel(file_path)

file_path2 = "/content/drive/MyDrive/TechLabs_Gruppe3/data_all_zweit.xlsx"
data_all_zweit = pd.read_excel(file_path2)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [8]:
!pip install dash plotly pandas dash_bootstrap_components

In [9]:
from dash import Dash, dcc, html, Input, Output
import dash_bootstrap_components as dbc
import plotly.express as px
import pandas as pd
import numpy as np
from flask import Flask

# Assuming data_all is your original DataFrame with the required data
# Transform Data to Long Format (Includes Erststimme and Zweitstimme)
data_all_long = (
    data_all
    .melt(id_vars=['wahljahr', 'gebiet'],
          value_vars=[col for col in data_all.columns if 'percent' in col],  # Only percent columns
          var_name='Partei_Stimme', value_name='Stimmen')
    .assign(
        Stimme=lambda df: df['Partei_Stimme'].str.extract(r'^(erst|zweit)')[0],  # Extract vote type (erst or zweit)
        Partei=lambda df: df['Partei_Stimme'].str.replace(r'^(erst_|zweit_)', '', regex=True)  # Extract party
    )
)

# Custom color palette for parties
party_colors = {
    'AFD': '#0489DB',
    'CDU': '#000000',
    'FDP': '#FFEF00',
    'GRÜNE': '#1AA037',
    'DIE LINKE': '#be3075',
    'SPD': '#e3000f'
}

# Define display names for parties
party_display_names = {
    "afd_percent": "AFD",
    "cdu_percent": "CDU",
    "fdp_percent": "FDP",
    "gruene_percent": "GRÜNE",
    "linke_percent": "DIE LINKE",
    "spd_percent": "SPD"
}

# Custom styles for the checklist
CHECKLIST_STYLE = {"margin-bottom": "10px", "font-weight": "bold"}
CARD_STYLE = {"padding": "10px", "border-radius": "5px", "background-color": "#f8f9fa"}

# Create Dash app with Bootstrap theme
server = Flask(__name__)
app = Dash(__name__, server=server, external_stylesheets=[dbc.themes.BOOTSTRAP])

# Layout of the app with filters and graph
app.layout = dbc.Container([
    html.H4('📊 Wahlverhalten in Münster (Prozentual)', className="mt-3 mb-4 text-center"),
    dcc.Graph(id="graph", style={"margin-bottom": "20px"}),  # Main graph

    dbc.Row([
        dbc.Col(dbc.Card([
            html.Label("🗳️ Wähle Stimmenart:", style=CHECKLIST_STYLE),
            dbc.Checklist(
                id="stimme-checklist",
                options=[{"label": "Erststimme", "value": "erst"}, {"label": "Zweitstimme", "value": "zweit"}],
                value=["zweit"],  # Default to only Zweitstimme
                inline=True,
                switch=True  # Switch toggle for vote type
            )
        ], style=CARD_STYLE), width=3),

        dbc.Col(dbc.Card([
            html.Label("📍 Wähle Region(en):", style=CHECKLIST_STYLE),
            dbc.Checklist(
                id="gebiet-checklist",
                options=[{"label": region, "value": region} for region in sorted(data_all_long['gebiet'].unique())],
                value=["Gesamt"],  # Default to "Gesamt"
                inline=True
            )
        ], style=CARD_STYLE), width=3),

        dbc.Col(dbc.Card([
            html.Label("🏛️ Wähle Partei(en):", style=CHECKLIST_STYLE),
            dbc.Checklist(
                id="party-checklist",
                options=[{"label": party_display_names.get(party, party), "value": party} for party in sorted(data_all_long['Partei'].unique())],
                value=sorted(data_all_long['Partei'].unique()),  # Default to all parties
                inline=True
            )
        ], style=CARD_STYLE), width=3),

        dbc.Col(dbc.Card([
            html.Label("📅 Wähle Jahr(e):", style=CHECKLIST_STYLE),
            dbc.Checklist(
                id='year-checklist',
                options=[{'label': str(year), 'value': year} for year in sorted(data_all_long['wahljahr'].unique())],
                value=sorted(data_all_long['wahljahr'].unique()),  # Default to all years
                inline=True
            )
        ], style=CARD_STYLE), width=3),
    ], className="mb-4")
])

# Callback function to update the graph based on selected filters
@app.callback(
    Output('graph', 'figure'),
    [Input('stimme-checklist', 'value'),
     Input('gebiet-checklist', 'value'),
     Input('party-checklist', 'value'),
     Input('year-checklist', 'value')]
)
def update_chart(selected_stimme, selected_regions, selected_parties, selected_years):
    # Efficient filtering using .query()
    filtered_df = data_all_long.query(
        "Stimme in @selected_stimme and gebiet in @selected_regions and Partei in @selected_parties and wahljahr in @selected_years"
    )

    # If no data, return a placeholder chart
    if filtered_df.empty:
        return px.line(title="Keine Daten für die ausgewählten Filter verfügbar")

    # Prevent pandas warnings and create a new column for display names
    filtered_df['Partei_Display'] = filtered_df['Partei'].map(party_display_names).fillna(filtered_df['Partei'])
    filtered_df = filtered_df.copy()
    filtered_df['Partei_Stimme_Gebiet'] = filtered_df['Partei_Display'] + " (" + filtered_df['Stimme'] + ") - " + filtered_df['gebiet']

    # Create the line chart
    fig = px.line(
        filtered_df,
        x='wahljahr',
        y='Stimmen',  # Using the percentage column 'Stimmen'
        color='Partei_Stimme_Gebiet',
        markers=True,
        color_discrete_map={party: party_colors.get(party.split(" (")[0], "#999999") for party in filtered_df['Partei_Stimme_Gebiet'].unique()},
    )

    # Adjust line styles for Erststimme and Zweitstimme
    for trace in fig.data:
        stimme_typ = trace.name.split(" (")[1].split(")")[0]  # Extract 'erst' or 'zweit'
        trace.line.dash = "solid" if stimme_typ == "zweit" else "dashdot"  # Dash-dot for Erststimme

    # Dynamic Y-axis scaling
    y_min = max(0, filtered_df['Stimmen'].min() - 5)
    y_max = min(100, filtered_df['Stimmen'].max() + 5)
    fig.update_layout(yaxis=dict(range=[y_min, y_max]))

    # Add annotations for the last data points
    last_points = filtered_df.loc[filtered_df.groupby(['Partei_Stimme_Gebiet'])['wahljahr'].idxmax()]

    for _, row in last_points.iterrows():
        fig.add_annotation(
            x=row['wahljahr'],
            y=row['Stimmen'],
            text=row['gebiet'],
            showarrow=False,
            yshift=10
        )

    # Final layout adjustments
    fig.update_layout(
        title="Wahlverhalten in Münster (Prozentual)",
        xaxis_title="Jahr",
        yaxis_title="Stimmenanteil (%)",
        xaxis=dict(tickmode='array', tickvals=sorted(data_all_long['wahljahr'].unique())),
        template="plotly_white"
    )

    return fig

# Start the Dash app
app.run_server(debug=False)



<IPython.core.display.Javascript object>